# Regression Baseline — Multi-Model Comparison

Compares **Ridge**, **RandomForest**, and **HistGradientBoosting** on remaining-time prediction.
Uses only log-based features (`BasicControlFlowFeatures`). Time-series features are excluded.

Checkpoints for the cleaned log and feature matrix are shared with the RandomForestRegressor PoC when `FEATURE_PARAMS` match.

In [1]:
import contextlib
import logging
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import RandomizedSearchCV
from tqdm import tqdm

from spi_time_series.pipeline.checkpointing import checkpoint
from spi_time_series.models.trainer import train
from spi_time_series.data import Dataset
from spi_time_series.data.constants import VALID_END_ACTIVITIES
from spi_time_series.data.schemas import EventLog, ModelArtifact, PreprocessedData, RawData
from spi_time_series.evaluation.metrics import evaluate
from spi_time_series.features.extraction import extract_features_builder
from spi_time_series.features.log_based_features import BasicControlFlowFeatures
from spi_time_series.preprocessing.preprocess import (
    _build_trace_samples,
    clean_data,
    sliding_window_factory,
    split_data,
)

logging.basicConfig(level=logging.INFO, force=True)

## Config

All tunable values live in the cells below.

In [2]:
RANDOM_STATE = 42
CV_FOLDS = 3
N_ITER = 10
MIN_PREFIX_LENGTH = 3
# Cap each trace at its MAX_PREFIX_LENGTH most recent events. Without a cap, BPI 2017
# generates ~800K training prefixes (avg trace length ~38, min_length=3). A cap of 10
# reduces this to ~220K and cuts extraction time proportionally.
MAX_PREFIX_LENGTH = 10

CHECKPOINT_DIR = Path("../data/checkpoints")

In [3]:
MODEL_CONFIGS = {
    "ridge": (
        Ridge(),
        # 10 alpha values so RandomizedSearchCV(n_iter=N_ITER) covers the full grid
        {"alpha": [1e-4, 1e-3, 1e-2, 1e-1, 1.0, 5.0, 10.0, 50.0, 100.0, 1000.0]},
    ),
    "random_forest": (
        RandomForestRegressor(n_jobs=-1, random_state=RANDOM_STATE),
        {
            "n_estimators": [50, 100, 150, 200],
            "max_depth": [None, 10, 20],
            "min_samples_split": [2, 5, 10],
            "max_features": ["sqrt", 0.5],
        },
    ),
    "hist_gradient_boosting": (
        HistGradientBoostingRegressor(random_state=RANDOM_STATE),
        {
            "max_iter": [100, 200],
            "max_depth": [None, 3, 5],
            "learning_rate": [0.05, 0.1, 0.2],
        },
    ),
}

In [4]:
CLEANED_PARAMS = {"valid_end_activities": sorted(VALID_END_ACTIVITIES)}

FEATURE_PARAMS = {
    **CLEANED_PARAMS,
    "min_prefix_length": MIN_PREFIX_LENGTH,
    "max_prefix_length": MAX_PREFIX_LENGTH,
    "feature_classes": [BasicControlFlowFeatures.__name__],
}

ARTIFACT_PARAMS = {
    **FEATURE_PARAMS,
    "models": {
        name: {"class": type(est).__name__, "hp_grid": grid}
        for name, (est, grid) in MODEL_CONFIGS.items()
    },
    "n_iter": N_ITER,
    "cv_folds": CV_FOLDS,
    "random_state": RANDOM_STATE,
}

## tqdm_joblib helper

Patches joblib's batch-completion callback so that every completed parallel job advances a tqdm bar.
No additional packages required — `tqdm` and `joblib` are already project dependencies.

In [5]:
@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old

## Target generator

The `TargetGenerator` protocol `(trace, start_idx, end_idx) -> float` has no `col_idx` parameter.
A module-level dict `_col_idx_store` is populated by the splitter (which runs before feature extraction)
and read by `remaining_time_hours` — safe because pipeline stages run sequentially.

In [6]:
_col_idx_store: dict[str, int] = {}


def remaining_time_hours(trace, start_idx: int, end_idx: int) -> float:
    ts_idx = _col_idx_store["time:timestamp"]
    delta = trace[-1, ts_idx] - trace[end_idx - 1, ts_idx]
    return delta.total_seconds() / 3600


def my_preprocessor(raw: RawData) -> EventLog:
    cleaned = clean_data(raw, valid_ends=VALID_END_ACTIVITIES)
    return cleaned.event_log


def my_splitter(log: EventLog) -> PreprocessedData:
    train_df, test_df = split_data(log)

    _col_idx_store.clear()
    _col_idx_store.update({c: i for i, c in enumerate(train_df.columns)})

    prefix_gen = sliding_window_factory(
        min_length=MIN_PREFIX_LENGTH,
        max_length=MAX_PREFIX_LENGTH,
    )

    return PreprocessedData(
        train_log=_build_trace_samples(train_df, prefix_gen),
        num_train_cases=len(train_df["case:concept:name"].unique()),
        test_log=_build_trace_samples(test_df, prefix_gen),
        num_test_cases=len(test_df["case:concept:name"].unique()),
        col_idx=_col_idx_store.copy(),
    )

## Stage 1 — Load & preprocess

In [7]:
dataset = Dataset()
raw = RawData(event_log=dataset.log)

cleaned = checkpoint(
    CHECKPOINT_DIR / "cleaned_log.pkl",
    lambda: my_preprocessor(raw),
    params=CLEANED_PARAMS,
)
preprocessed = my_splitter(cleaned)

INFO:spi_time_series.data.dataset:Dataset found at G:\spi-time-series\data\raw\BPI Challenge 2017.xes.gz.
INFO:spi_time_series.data.dataset:Reading XES log...
g:\spi-time-series\.venv\Lib\site-packages\pm4py\utils.py:1005: UserWarning: In the current version, the import/export operation uses `r4pm` by default for importing/exporting files faster.
  warnings.warn(
INFO:spi_time_series.data.dataset:Done. Loaded 1202267 events.
INFO:spi_time_series.pipeline.checkpointing:Loading checkpoint: ..\data\checkpoints\cleaned_log__9ea2b74a.pkl
INFO:spi_time_series.preprocessing.preprocess:Splitting cases at cutoff time: 2016-10-18 17:03:13.107600128+00:00
INFO:spi_time_series.preprocessing.preprocess:Split successful. 
Train: 22723 cases, 
Test: 6247 cases.


## Stage 2 — Feature extraction

`one_hot_encode_categorical` is left **off** (default). The bag-of-activities columns (`count_<activity>`)
already capture which activities appeared and how many times, so the information loss is acceptable for a baseline.

Checkpoint is shared with the RandomForestRegressor PoC when `FEATURE_PARAMS` match.

In [8]:
feature_extractor = extract_features_builder(
    [BasicControlFlowFeatures()],
    remaining_time_hours,
)

features = checkpoint(
    CHECKPOINT_DIR / "features.pkl",
    lambda: feature_extractor(preprocessed),
    params=FEATURE_PARAMS,
)

print(f"Train shape: {features.X_train.shape}")
print(f"Test shape:  {features.X_test.shape}")

INFO:spi_time_series.pipeline.checkpointing:Loading checkpoint: ..\data\checkpoints\features__c9afd2c5.pkl


Train shape: (822706, 33)
Test shape:  (227073, 33)


## Stages 3 & 4 — Hyperparameter search + fit

For each model: randomised CV search (`refit=False`) prints best params and CV MAE,
then `train()` fits every best estimator inside a full preprocessing pipeline.
All three models are wrapped in `_fit_artifact()` so the entire block is skipped when the artifact checkpoint exists.

In [16]:
def _fit_artifact() -> ModelArtifact:
    search_pre = ColumnTransformer(
        [("num", SimpleImputer(strategy="median"), make_column_selector(dtype_include=np.number))],
        remainder="drop",
    )
    X_train_num = search_pre.fit_transform(features.X_train)

    best_estimators = {}
    for name, (base_estimator, hp_grid) in MODEL_CONFIGS.items():
        print(f"\n--- {name} ---")
        search = RandomizedSearchCV(
            base_estimator,
            param_distributions=hp_grid,
            n_iter=N_ITER,
            cv=CV_FOLDS,
            scoring="neg_mean_absolute_error",
            refit=False,
            random_state=RANDOM_STATE,
        )
        with tqdm_joblib(tqdm(desc=f"Search: {name}", total=None)):
            search.fit(X_train_num, features.y_train)
        print(f"  best params: {search.best_params_}")
        print(f"  CV MAE:      {-search.best_score_:.2f} h")
        best_estimators[name] = clone(base_estimator).set_params(**search.best_params_)

    print("\nFitting final models on full training set...")
    return train(features, best_estimators)

In [17]:
artifact = checkpoint(CHECKPOINT_DIR / "artifact.pkl", _fit_artifact, params=ARTIFACT_PARAMS)

INFO:spi_time_series.pipeline.checkpointing:Checkpoint not found — computing: ..\data\checkpoints\artifact__2fbd90c5.pkl



--- ridge ---


Search: ridge: 0it [00:00, ?it/s]

  best params: {'alpha': 1000.0}
  CV MAE:      183.76 h

--- random_forest ---


Search: ridge: 0it [00:15, ?it/s]


  best params: {'n_estimators': 150, 'min_samples_split': 2, 'max_features': 0.5, 'max_depth': 20}
  CV MAE:      176.22 h

--- hist_gradient_boosting ---


Search: hist_gradient_boosting: 900it [01:20, 12.65it/s]INFO:spi_time_series.models.trainer:Training model: ridge


  best params: {'max_iter': 200, 'max_depth': None, 'learning_rate': 0.2}
  CV MAE:      176.52 h

Fitting final models on full training set...


INFO:spi_time_series.models.trainer:Model ridge trained.
INFO:spi_time_series.models.trainer:Training model: random_forest
Search: hist_gradient_boosting: 930it [01:33, 12.65it/s]INFO:spi_time_series.models.trainer:Model random_forest trained.
INFO:spi_time_series.models.trainer:Training model: hist_gradient_boosting
INFO:spi_time_series.models.trainer:Model hist_gradient_boosting trained.
INFO:spi_time_series.pipeline.checkpointing:Checkpoint saved: ..\data\checkpoints\artifact__2fbd90c5.pkl


## Stage 5 — Evaluate

In [18]:
report = evaluate(artifact, features)

INFO:spi_time_series.evaluation.metrics:Evaluating model: ridge
INFO:spi_time_series.evaluation.metrics:Model ridge evaluated across 8 prefix lengths.
INFO:spi_time_series.evaluation.metrics:Evaluating model: random_forest
INFO:spi_time_series.evaluation.metrics:Model random_forest evaluated across 8 prefix lengths.
INFO:spi_time_series.evaluation.metrics:Evaluating model: hist_gradient_boosting
INFO:spi_time_series.evaluation.metrics:Model hist_gradient_boosting evaluated across 8 prefix lengths.


## Results

MAE, RMSE (hours), and R² per model and prefix length.

In [19]:
rows = [
    {
        "model": model_name,
        "prefix_length": pl,
        "MAE (h)": round(report.metrics[model_name][pl]["mae"], 3),
        "RMSE (h)": round(report.metrics[model_name][pl]["rmse"], 3),
        "R\u00b2": round(report.metrics[model_name][pl]["r2"], 4),
    }
    for model_name in sorted(report.model_names)
    for pl in report.prefix_lengths
]

pd.DataFrame(rows)

,model,prefix_length,MAE (h),RMSE (h),R²
0,hist_gradient_boosting,3,248.798,310.368,-0.1708
1,hist_gradient_boosting,4,240.186,283.638,0.0214
2,hist_gradient_boosting,5,239.879,283.864,0.0198
3,hist_gradient_boosting,6,239.831,283.585,0.0231
4,hist_gradient_boosting,7,237.772,281.960,0.0218
5,hist_gradient_boosting,8,237.249,280.587,0.0313
6,hist_gradient_boosting,9,235.506,279.364,0.0297
7,hist_gradient_boosting,10,157.198,219.712,0.3859
8,random_forest,3,240.494,283.130,0.0257
9,random_forest,4,241.647,285.333,0.0096


### MAE comparison (pivot)

Rows = prefix length, columns = model.

In [20]:
df_mae = pd.DataFrame(
    {
        model_name: {
            pl: round(report.metrics[model_name][pl]["mae"], 1)
            for pl in report.prefix_lengths
        }
        for model_name in sorted(report.model_names)
    }
)
df_mae.index.name = "prefix_length"
df_mae

,hist_gradient_boosting,random_forest,ridge
prefix_length,,,
3,248.8,240.5,242.6
4,240.2,241.6,242.1
5,239.9,241.0,241.2
6,239.8,240.1,241.8
7,237.8,238.2,241.2
8,237.2,237.8,241.0
9,235.5,237.2,240.3
10,157.2,157.1,164.2
